# Notebook 063: Streaming Observability: StreamingQueryListener and SLA Latency Alerting

**Module:** Module 06: Streaming & Near Real-Time Processing  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 10: Databricks Jobs & Video 12: Debugging Code in Databricks  

---

## 1. Business Scenario & Real-World Context
An enterprise fintech streaming pipeline must maintain an SLA of processing events within 60 seconds of origination. If a backlog builds up or micro-batch processing time spikes, on-call engineers must be alerted before SLA violations occur.

## 2. Technical Requirement & Problem Statement
Implement streaming observability in PySpark: register a custom `StreamingQueryListener`, capture query startup, termination, and micro-batch progress events (`onQueryProgress`), calculate input rate vs processing rate, and generate automated SLA breach alerts.

## 3. Spark & Databricks Architecture Notes
Spark's `StreamingQueryListener` provides an asynchronous callback API. When registered with `spark.streams.addListener()`, it captures event telemetry for all queries running in the SparkSession without introducing runtime overhead into the data processing tasks.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_063_Streaming_Observability_StreamingQueryListener_and_SLA_Latency_Alerting")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Defining Custom StreamingQueryListener Class
Implement listener to intercept query progress and measure SLA metrics.

In [ ]:
from pyspark.sql.streaming import StreamingQueryListener

class PipelineSLAListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"[OBSERVABILITY] Query Started! ID: {event.id} | Name: {event.name}")

    def onQueryProgress(self, event):
        progress = event.progress
        batch_id = progress.batchId
        input_rate = progress.inputRowsPerSecond
        proc_rate = progress.processedRowsPerSecond
        num_input = progress.numInputRows
        
        # Check if processing rate lags behind input rate (Backlog accumulation!)
        status = "HEALTHY"
        if input_rate > 0 and proc_rate < (input_rate * 0.8):
            status = "WARNING: PROCESSING_LAG_DETECTED"
            
        print(f"[METRIC] Batch: {batch_id} | In: {num_input} rows | InRate: {input_rate:.1f}/s | ProcRate: {proc_rate:.1f}/s | {status}")

    def onQueryTerminated(self, event):
        if event.exception:
            print(f"[ALERT] Query CRASHED with error: {event.exception}")
        else:
            print("[OBSERVABILITY] Query Terminated cleanly.")

print("PipelineSLAListener class defined.")

## 6. Registering Listener with SparkSession Streams Manager
Register the listener to begin capturing streaming telemetry.

In [ ]:
# Register the listener
sla_listener = PipelineSLAListener()
spark.streams.addListener(sla_listener)
print("StreamingQueryListener registered with SparkSession.")

## 7. Running Diagnostic Stream to Trigger Listener Callbacks
Execute a short rate stream to observe listener telemetry in the logs.

In [ ]:
import time

df_rate = spark.readStream.format("rate").option("rowsPerSecond", 10).load()

q_monitored = (
    df_rate.writeStream
    .format("memory")
    .queryName("sla_telemetry_demo")
    .start()
)

# Let run for 2.5 seconds to trigger progress callbacks
time.sleep(2.5)

q_monitored.stop()
print("Monitored query completed.")

## 9. Key Architectural Takeaways
- **`StreamingQueryListener` provides non-invasive, asynchronous observability into all streaming queries.**
- **Monitor `inputRowsPerSecond` vs `processedRowsPerSecond`: when processing rate drops below input rate, backpressure develops.**
- **Send listener alerts to Slack, PagerDuty, or Datadog to detect silent streaming stalls.**

## 10. Senior Data Engineering Interview Questions
1. What are the three core callback methods in Spark's `StreamingQueryListener`?
2. How do you detect backpressure in a Spark Structured Streaming query programmatically?
3. What is the difference between `triggerExecution` duration and `batchDuration` in streaming metrics?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
